# finance / portfolio / return

**Intent**
- Outputs `finance_portfolio_return_nominal_{low,base,high}_pct`, the nominal annual return to 2070.
- Gives low, base and high scenarios as compound rates in percent per year, as constants.
- Exists to grow the portfolio in the plan.

**Data**
- Personal data is git-ignored in `data/` and `project/`, with synthetic placeholders committed.
- `training` is ten years of annual portfolio returns.
- It is synthetic until the `wrangle` series is loaded.
- The `equity` table holds prices and changes, but portfolio weights are only in the Portfolio sheet.

**Status**
- The base is a placeholder, the mean annual return.
- Low and high are one standard deviation either side.
- It carries no evaluation, so the golden replay is the gate.

**Future**
- Add `query_postgres` to `workbench.data` and load the real series in place of the synthetic ones.
- Land the portfolio weights in Postgres, and use total return, not price return.
- Blend the history with forward capital market assumptions, since 15 years cannot tell 6 percent from 7.
- Bootstrap monthly returns in blocks to 2070, re-centred on the forward base, and take the compound rate.
- Set base as the median and low and high as the 25th and 75th percentiles.
- Give the real return by subtracting the matching `household/costs` growth, and allow for fees and tax.
- Serve the numbers to the planning sheet through `wrangle` and an Apps Script function, which is not yet built.

## Setup ([6.7.1](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#671-setup))

In [ ]:
import json

from workbench.data import ensure_data, golden_data
from workbench.env import init_env
from workbench.project import load_signature, provenance, record_project, scratch_project_dir, signature_names

env = init_env()

WORKSPACE = "finance/portfolio/return"
MODEL_NAME = "finance_portfolio_return"
TARGET = {"kind": "served"}
signature = load_signature(WORKSPACE)
INPUTS, OUTPUTS = signature_names(signature)
TRAINING = "portfolio_return_training_2026-11-05_2026-11-05"
GOLDEN = "portfolio_return_golden"
INPUT = INPUTS[0]

## Capture Data ([6.7.2](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#672-capture-data))

In [ ]:
DEVIATIONS = [-4.0, -3.0, -2.0, -1.0, 0.0, 0.0, 1.0, 2.0, 3.0, 4.0]

training_pl, training_path = ensure_data(WORKSPACE, "training", TRAINING, lambda: {"year": list(range(2016, 2026)), "portfolio_return_pct": [round(6.0 + deviation, 4) for deviation in DEVIATIONS]})
print("training", training_pl.height)

## Fit ([6.7.4](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#674-fit))

In [ ]:
BASE = round(float(training_pl["portfolio_return_pct"].mean()), 2)
SPREAD = round(float(training_pl["portfolio_return_pct"].std(ddof=0)), 2)
CONSTANTS = dict(zip(OUTPUTS, [round(BASE - SPREAD, 2), BASE, round(BASE + SPREAD, 2)], strict=True))
print("constants", CONSTANTS)
assert list(CONSTANTS.values()) == [3.55, 6.0, 8.45], "the fit did not recover the mean and spread"

## Package ([6.7.6](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#676-package))

In [ ]:
from workbench.model import package_project, register_preview
from workbench.replay import require_replay

golden_pl, golden_path = golden_data(WORKSPACE, GOLDEN, lambda: {INPUT: [1.0, None], "expect": ["ok", "error"]})

project_dir = scratch_project_dir(WORKSPACE)
manifest = package_project(
    project_dir,
    MODEL_NAME,
    "static",
    signature,
    golden_pl.iter_rows(named=True),
    provenance(
        "manual",
        "synthetic annual portfolio returns, training split",
        [training_path, golden_path],
        "mean and standard deviation of annual returns",
    ),
    constants=CONSTANTS,
)
print(json.dumps(manifest, indent=2))

require_replay(project_dir)
metadata = record_project(WORKSPACE, project_dir, TARGET)
print("replayed the golden sequences exactly through the shipped loader")

## Register ([6.7.7](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#677-register))

In [ ]:
version = register_preview(project_dir, MODEL_NAME, env["MLFLOW_TRACKING_URI"], TARGET, metadata["version"])
print("registered version", version)